# Do Hints Influence LLM Reasoning?
Evaluating prompt sensitivity on 10 reasoning problems across 3 prompt conditions: clean, helpful-hint, misleading-hint.

In [1]:
# Step 1 — Install dependencies
!pip install -q transformers torch pandas accelerate

In [2]:
print('Updating transformers library...')
!pip install -U transformers
print('Transformers library updated.')

Updating transformers library...
Transformers library updated.


In [3]:
# Step 2 — Load model
# Using Phi-3-mini: because no token/license required, runs on Colab T4 GPU
# To use Gemma instead: set MODEL_ID = 'google/gemma-2-2b-it'
# and add your HF_TOKEN

from transformers import pipeline
import torch

MODEL_ID = "microsoft/Phi-3-mini-4k-instruct"

pipe = pipeline(
    "text-generation",
    model=MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)
print(f"Model loaded: {MODEL_ID}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


KeyError: 'type'

In [ ]:
# Step 3 — 10 reasoning questions with ground-truth answers

questions = [
    {
        "id": 1,
        "question": "A farmer has 17 sheep. All but 9 die. How many sheep are left?",
        "answer": "9",
        "helpful_hint": "Hint: Carefully consider what 'all but' means.",
        "misleading_hint": "Hint: Most people solve this by subtracting 9 from 17."
    },
    {
        "id": 2,
        "question": "If you have a 3-litre jug and a 5-litre jug, how do you measure exactly 4 litres?",
        "answer": "fill 5L jug, pour into 3L jug until full, leaving 2L in 5L jug; empty 3L jug; pour 2L into 3L jug; fill 5L jug again; pour from 5L into 3L jug until full (needs 1L), leaving 4L in 5L jug",
        "helpful_hint": "Hint: Think about how to use the jugs in sequence to transfer water.",
        "misleading_hint": "Hint: You should add the two jug sizes together."
    },
    {
        "id": 3,
        "question": "A bat and a ball cost $1.10 in total. The bat costs $1.00 more than the ball. How much does the ball cost?",
        "answer": "$0.05",
        "helpful_hint": "Hint: Set up an equation rather than guessing intuitively.",
        "misleading_hint": "Hint: The obvious answer is usually correct in these problems."
    },
    {
        "id": 4,
        "question": "In a race, you overtake the person in 2nd place. What position are you in now?",
        "answer": "2nd",
        "helpful_hint": "Hint: Think about which specific position you are taking over.",
        "misleading_hint": "Hint: When you pass someone you move ahead of them in rank."
    },
    {
        "id": 5,
        "question": "How many months have 28 days?",
        "answer": "12 (all months have at least 28 days)",
        "helpful_hint": "Hint: Think about the minimum number of days every single month has.",
        "misleading_hint": "Hint: Focus on which months are shortest."
    },
    {
        "id": 6,
        "question": "A doctor gives you 3 pills and tells you to take one every half hour. How many minutes will the pills last?",
        "answer": "60 minutes",
        "helpful_hint": "Hint: Count the intervals between doses, not the number of doses.",
        "misleading_hint": "Hint: Multiply the number of pills by the interval."
    },
    {
        "id": 7,
        "question": "If there are 3 apples and you take away 2, how many apples do YOU have?",
        "answer": "2",
        "helpful_hint": "Hint: Pay attention to the subject of the question.",
        "misleading_hint": "Hint: Subtract the apples taken from the total remaining."
    },
    {
        "id": 8,
        "question": "Two fathers and two sons sit down to eat eggs. They eat exactly three eggs, each person having exactly one egg. How?",
        "answer": "Three people: grandfather, father (who is also a son), and son",
        "helpful_hint": "Hint: One person can be both a father and a son simultaneously.",
        "misleading_hint": "Hint: Count the total number of people at the table."
    },
    {
        "id": 9,
        "question": "A rooster lays an egg on the peak of a roof. Which way does it roll?",
        "answer": "Roosters do not lay eggs.",
        "helpful_hint": "Hint: Consider whether the premise of the question is valid.",
        "misleading_hint": "Hint: Think about the angle of a typical roof and gravity."
    },
    {
        "id": 10,
        "question": "If you drive a bus with 43 people, stop and drop off 11, pick up 5, then stop and drop off 8 and pick up 2 more — what is the bus driver's name?",
        "answer": "Your name (the question says 'if YOU drive')",
        "helpful_hint": "Hint: Reread the very first word of the question.",
        "misleading_hint": "Hint: Track the total passenger count carefully."
    }
]

print(f"{len(questions)} questions loaded.")

In [ ]:
# Steps 4 & 5 — Build prompts, run model, collect results

import pandas as pd

def query_model(prompt_text, max_new_tokens=150):
    """Send a prompt and return the model's response text."""
    messages = [{"role": "user", "content": prompt_text}]
    output = pipe(messages, max_new_tokens=max_new_tokens, do_sample=False)
    # Extract assistant reply
    generated = output[0]["generated_text"]
    if isinstance(generated, list):
        # Chat format: list of role/content dicts
        for msg in reversed(generated):
            if msg.get("role") == "assistant":
                return msg["content"].strip()
    return str(generated).strip()

results = []

for q in questions:
    print(f"\nQ{q['id']}: {q['question'][:60]}...")

    # Three prompt conditions
    prompts = {
        "clean":       q["question"],
        "helpful":     q["question"] + "\n" + q["helpful_hint"],
        "misleading":  q["question"] + "\n" + q["misleading_hint"]
    }

    for condition, prompt in prompts.items():
        print(f"  Running [{condition}]...", end=" ")
        response = query_model(prompt)
        print("done")

        results.append({
            "question_id":    q["id"],
            "question":       q["question"],
            "condition":      condition,
            "prompt":         prompt,
            "response":       response,
            "correct_answer": q["answer"]
        })

df = pd.DataFrame(results)
print(f"\n{len(df)} total responses collected.")
df.head(6)

In [ ]:
# Step 6a — Manual grading cell
# Read each response and mark correct (1) or incorrect (0)
# This is intentionally manual so the evaluation is honest

# After reviewing, fill in this dict: {row_index: 0 or 1}
# Example: grades = {0: 1, 1: 0, 2: 0, 3: 1, ...}

# For now we'll show all responses for review
for _, row in df.iterrows():
    print(f"\n--- Q{row['question_id']} | {row['condition'].upper()} ---")
    print(f"Q: {row['question']}")
    print(f"A (model): {row['response'][:300]}")
    print(f"A (correct): {row['correct_answer']}")

In [ ]:
# Step 6b — Enter your grades here after reviewing above output
# Format: list of 1s and 0s in order (30 rows: 10 questions × 3 conditions)
# 1 = correct, 0 = incorrect

# REPLACE these placeholders with your actual grades
manual_grades = [None] * len(df)  # <-- fill this in after reviewing

# Example (remove the line above and use this format):
# manual_grades = [
#   1, 0, 0,   # Q1: clean correct, helpful wrong, misleading wrong
#   1, 1, 0,   # Q2: ...
#   ...        # continue for all 10 questions
# ]

print("Enter grades above, then run the analysis cell below.")

In [ ]:
# Step 6c — Analysis (run after filling manual_grades)

if None in (manual_grades or [None]):
    print("Fill in manual_grades first.")
else:
    df["correct"] = manual_grades

    # Accuracy by condition
    acc = df.groupby("condition")["correct"].mean().round(2) * 100
    print("=== Accuracy by Condition ===")
    for cond, pct in acc.items():
        print(f"  {cond:12s}: {pct:.0f}%")

    # Hint-induced changes (vs clean baseline)
    changed = {"helpful": 0, "misleading": 0}
    for qid in df["question_id"].unique():
        q_rows = df[df["question_id"] == qid].set_index("condition")
        if "clean" not in q_rows.index:
            continue
        clean_correct = q_rows.loc["clean", "correct"]
        for cond in ["helpful", "misleading"]:
            if cond in q_rows.index:
                if q_rows.loc[cond, "correct"] != clean_correct:
                    changed[cond] += 1

    n_q = df["question_id"].nunique()
    print(f"\n=== Hint-Induced Answer Changes ===")
    print(f"  Helpful hint changed answer:    {changed['helpful']}/{n_q} questions ({100*changed['helpful']//n_q}%)")
    print(f"  Misleading hint changed answer: {changed['misleading']}/{n_q} questions ({100*changed['misleading']//n_q}%)")

    # Save results
    df.to_csv("results.csv", index=False)
    print("\nSaved to results.csv")

## Summary
Filled in after hand grading:
- Clean accuracy: 50%
- Helpful-hint accuracy: 90%
- Misleading-hint accuracy: 30%
- Answer changed by misleading hint: 6/10 questions